[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/llm-course/blob/main/slides/week6/bert_demo.ipynb)

# BERT deep dive — interactive exploration

**PSYC 51.17: Models of language and communication**  
**Week 6**

---

## Learning objectives

By the end of this session, you will:
1. Use BERT to predict masked words and explore what it "knows"
2. Visualize attention patterns to see how BERT reads sentences
3. Compare how word representations change across BERT's layers
4. Discover biases and world knowledge encoded in BERT's weights

## Setup

In [ ]:
# Install required packages (for Colab)
!pip install -q transformers torch sentence-transformers matplotlib numpy

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import norm
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
    BertTokenizer,
    BertModel,
)
import warnings
warnings.filterwarnings('ignore')

print("\u2713 All imports successful!")

## Part 1: Fill-in-the-blank predictions

BERT's main training task was **Masked Language Modeling (MLM)**: predict a hidden word from its surrounding context. Let's see what BERT learned from 3.3 billion words of text.

In [ ]:
# Load the fill-mask pipeline
fill_mask = pipeline("fill-mask", model="bert-base-uncased")

# Try a simple sentence
results = fill_mask("The [MASK] barked at the mailman.")
print("The [MASK] barked at the mailman.")
print("-" * 40)
for r in results:
    print(f"  {r['token_str']:>12s}  ({r['score']:.3f})")

In [ ]:
# BERT learned facts about the world!
sentences = [
    "The capital of France is [MASK].",
    "Water freezes at [MASK] degrees.",
    "The largest planet in our solar system is [MASK].",
    "Shakespeare wrote Romeo and [MASK].",
]

for sent in sentences:
    results = fill_mask(sent)
    top = results[0]
    print(f"{sent}")
    print(f"  \u2192 {top['token_str']} ({top['score']:.3f})")
    print()

In [ ]:
# BERT reveals biases from its training data
bias_sentences = [
    "The nurse said [MASK] would be right back.",
    "The doctor said [MASK] would be right back.",
    "The engineer said [MASK] would be right back.",
    "The teacher said [MASK] would be right back.",
]

print("Gender pronouns in BERT predictions:")
print("=" * 55)
for sent in bias_sentences:
    results = fill_mask(sent)
    scores = {}
    for r in results:
        if r['token_str'] in ('he', 'she'):
            scores[r['token_str']] = r['score']

    profession = sent.split("The ")[1].split(" said")[0]
    he_score = scores.get('he', 0)
    she_score = scores.get('she', 0)
    print(f"  {profession:>10s}:  he={he_score:.3f}  she={she_score:.3f}")

In [ ]:
# YOUR TURN! Try your own sentences.
# Replace one word with [MASK] and see what BERT predicts.

my_sentence = "I love eating [MASK] for breakfast."
results = fill_mask(my_sentence)
print(f"Sentence: {my_sentence}")
print("Top 5 predictions:")
for r in results:
    print(f"  {r['token_str']:>12s}  ({r['score']:.3f})")

### 💡 Discussion

- What surprised you about BERT's predictions?
- Did BERT get any factual questions wrong? What does that tell us?
- Look at the gender bias results. Where do these biases come from?
- Try a sentence in a domain BERT probably hasn't seen much of (e.g., TikTok slang). What happens?

## Part 2: Attention visualization

BERT has **12 layers × 12 attention heads = 144** different "views" of each sentence. Some heads specialize in specific linguistic relations (direct objects, coreference, etc.). Let's look inside.

In [ ]:
# Load model with attention outputs
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

sentence = "The cat sat on the mat because it was comfortable."
inputs = tokenizer(sentence, return_tensors="pt")
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

with torch.no_grad():
    outputs = model(**inputs)

# Visualize attention from layer 8, head 10 (often handles coreference)
attention = outputs.attentions[7][0, 9].numpy()  # layer 8 (idx 7), head 10 (idx 9)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attention, cmap="Blues")
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
ax.set_xlabel("Attended to", fontsize=12)
ax.set_ylabel("Attending from", fontsize=12)
ax.set_title("BERT attention (layer 8, head 10)", fontsize=14)
plt.colorbar(im)
plt.tight_layout()
plt.show()

# What does "it" attend to?
it_idx = tokens.index("it")
print(f"\nToken 'it' attends most strongly to:")
for i, score in sorted(enumerate(attention[it_idx]), key=lambda x: -x[1])[:5]:
    print(f"  '{tokens[i]}': {score:.3f}")

In [ ]:
# Compare what different heads focus on
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
interesting_heads = [
    (0, 0, "Layer 1, head 1\n(often positional)"),
    (2, 3, "Layer 3, head 4\n(often syntactic)"),
    (5, 3, "Layer 6, head 4\n(often coreference)"),
    (7, 9, "Layer 8, head 10\n(often coreference)"),
    (9, 6, "Layer 10, head 7\n(often semantic)"),
    (11, 0, "Layer 12, head 1\n(task-oriented)"),
]

for ax, (layer, head, title) in zip(axes.flat, interesting_heads):
    attn = outputs.attentions[layer][0, head].numpy()
    ax.imshow(attn, cmap="Blues")
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(tokens, fontsize=8)
    ax.set_title(title, fontsize=11)

plt.suptitle(f'BERT attention patterns across layers\n"{sentence}"', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 💡 Discussion

- What patterns do you notice across layers? Do early layers look different from later ones?
- Do some heads look at neighboring tokens while others look at distant ones?
- Can you find a head where "it" strongly attends to "cat"?
- Try changing the sentence to something with more complex pronoun references.

## Part 3: Layer-by-layer embeddings

BERT's 12 layers progressively transform word representations from **surface features** to **deep semantic understanding**. The same word gets a completely different vector depending on context — and this difference grows across layers.

In [ ]:
# Load model with hidden state outputs
model_layers = BertModel.from_pretrained("bert-base-uncased", output_hidden_states=True)
tokenizer_layers = BertTokenizer.from_pretrained("bert-base-uncased")

def get_word_embeddings(sentence, target_word):
    """Extract embeddings for a target word from all 13 layers (embedding + 12 transformer)."""
    inputs = tokenizer_layers(sentence, return_tensors="pt")
    tokens = tokenizer_layers.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        outputs = model_layers(**inputs)

    # Find index of target word
    target_tokens = tokenizer_layers.tokenize(target_word)
    target_id = tokens.index(target_tokens[0])

    # Get embedding from each layer
    embeddings = [layer[0, target_id].numpy() for layer in outputs.hidden_states]
    return embeddings, tokens

# Two different meanings of "cell"
sent1 = "The prisoner escaped from the cell."
sent2 = "The biologist studied the cell under a microscope."

emb1, _ = get_word_embeddings(sent1, "cell")
emb2, _ = get_word_embeddings(sent2, "cell")

# Cosine similarity across layers
similarities = []
for e1, e2 in zip(emb1, emb2):
    sim = np.dot(e1, e2) / (norm(e1) * norm(e2))
    similarities.append(sim)

# Plot
plt.figure(figsize=(10, 5))
layers = list(range(13))
plt.plot(layers, similarities, "o-", color="#00693e", linewidth=2, markersize=8)
plt.xlabel("Layer", fontsize=12)
plt.ylabel("Cosine similarity", fontsize=12)
plt.title('How "cell" diverges across layers\n(prison vs biology context)', fontsize=14)
plt.xticks(layers, ["Emb"] + [str(i) for i in range(1, 13)])
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Layer  0 (embedding): similarity = {similarities[0]:.3f}")
print(f"Layer  6 (middle):    similarity = {similarities[6]:.3f}")
print(f"Layer 12 (final):     similarity = {similarities[12]:.3f}")

In [ ]:
# Try several ambiguous words
word_pairs = [
    ("He deposited money at the bank.", "The river bank was covered in moss.", "bank"),
    ("She played a note on the piano.", "He left a note on the fridge.", "note"),
    ("The bat flew out of the cave.", "He swung the bat at the ball.", "bat"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ["#00693e", "#267aba", "#9d162e"]

for ax, (s1, s2, word), color in zip(axes, word_pairs, colors):
    e1, _ = get_word_embeddings(s1, word)
    e2, _ = get_word_embeddings(s2, word)
    sims = [np.dot(a, b) / (norm(a) * norm(b)) for a, b in zip(e1, e2)]

    ax.plot(range(13), sims, "o-", color=color, linewidth=2, markersize=6)
    ax.set_title(f'"{word}"', fontsize=13)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Cosine similarity")
    ax.set_ylim(0, 1)
    ax.set_xticks(range(13))
    ax.set_xticklabels(["E"] + [str(i) for i in range(1, 13)], fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Word sense disambiguation across BERT layers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 💡 Discussion

- At which layer does BERT start distinguishing different word senses? Is it the same layer for all words?
- Why does similarity start high and decrease? What does the embedding layer "know" vs what it doesn't?
- Try adding your own ambiguous word pair. Does the pattern hold?

## Part 4: Sentence similarity

BERT can also compute **sentence-level embeddings** for measuring semantic similarity. We'll use `sentence-transformers`, which wraps BERT with a pooling layer optimized for comparing sentences.

In [ ]:
from sentence_transformers import SentenceTransformer, util

st_model = SentenceTransformer("all-MiniLM-L6-v2")  # Small, fast, effective

sentences = [
    "The cat is sleeping on the couch.",
    "A feline is resting on the sofa.",
    "The stock market crashed yesterday.",
    "Dogs are playing in the park.",
    "Financial markets experienced a downturn.",
]

embeddings = st_model.encode(sentences)
sim_matrix = util.cos_sim(embeddings, embeddings).numpy()

# Plot similarity matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(sentences)))
ax.set_yticks(range(len(sentences)))
short_labels = [s[:30] + "..." if len(s) > 30 else s for s in sentences]
ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

for i in range(len(sentences)):
    for j in range(len(sentences)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Sentence similarity matrix", fontsize=14)
plt.colorbar(im, label="Cosine similarity")
plt.tight_layout()
plt.show()

In [ ]:
# YOUR TURN! Add your own sentences and see how similar they are.
my_sentences = [
    "I love programming in Python.",
    "Python is my favorite language for coding.",
    "The python snake is quite large.",
    # Add more sentences here!
]

my_embeddings = st_model.encode(my_sentences)
my_sims = util.cos_sim(my_embeddings, my_embeddings).numpy()

for i in range(len(my_sentences)):
    for j in range(i + 1, len(my_sentences)):
        s1_short = my_sentences[i][:40]
        s2_short = my_sentences[j][:40]
        print(f"'{s1_short}' \u2194 '{s2_short}'")
        print(f"  Similarity: {my_sims[i][j]:.3f}")
        print()

### 💡 Discussion

- Which sentence pairs are most similar? Does this match your intuition?
- The "Python" example shows the same word in programming vs animal contexts. How well does the model separate them?
- Try sentences that are semantically similar but use completely different words (e.g., "The movie was terrible" vs "I hated that film").

## Summary

| Part | What we explored | Key insight |
|------|-----------------|-------------|
| Fill-in-the-blank | BERT predicts masked words | BERT learned grammar, facts, AND biases |
| Attention | How BERT reads sentences | Different heads specialize in different linguistic relations |
| Layer embeddings | How representations change | Lower layers = surface, upper = semantics |
| Sentence similarity | Meaning comparison | Semantically similar sentences cluster together |

## Further exploration

1. **Multilingual BERT**: Try `fill_mask` with `bert-base-multilingual-cased` — does it work in other languages?
2. **Attention archaeology**: Find an attention head that consistently links verbs to their subjects across different sentences.
3. **Layer depth**: Compare BERT-base (12 layers) vs BERT-large (24 layers) — does the word sense divergence happen at the same relative depth?
4. **Similarity limits**: Try sentence similarity with very long paragraphs — does it still capture meaning well?